In [1]:
import os
import random
from typing import List

import pandas as pd
import scipy.io
from scipy.io import arff
from numpy import load
from sklearn import preprocessing

In [ ]:
# -------------------------
# Utility / preprocessing
# -------------------------
def normalize(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize feature columns (all columns except the last) to range [0,1].
    The last column is assumed to be the target/outlier label and is kept unchanged.
    """
    if df.shape[1] < 2:
        return df.copy()
    features = df.iloc[:, :-1].values
    scaler = preprocessing.MinMaxScaler()
    features_scaled = scaler.fit_transform(features)
    df_new = pd.DataFrame(features_scaled, columns=df.columns.tolist()[:-1])
    df_new[df.columns.tolist()[-1]] = df.iloc[:, -1].values
    return df_new


def remove_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy of DataFrame with duplicate rows removed."""
    return df.drop_duplicates().reset_index(drop=True)


# -------------------------
# Outlier handling
# -------------------------
def limit_outlier_fraction(df: pd.DataFrame, percent: float, versions: int = 10) -> List[pd.DataFrame]:
    """
    Create up to `versions` variations of `df` where the fraction of rows labeled
    as outlier (assumes column named 'outlier' with values 'yes'/'no' or 1/0) does not exceed `percent`.
    If the current fraction is already <= percent, a single copy is returned.
    """
    datas = []
    # Ensure outlier column standardized to 'yes'/'no'
    if 'outlier' in df.columns:
        df_proc = df.copy(deep=True)
        df_proc['outlier'] = df_proc['outlier'].apply(lambda x: 'yes' if (x == 1 or (isinstance(x, bytes) and x == b'1') or str(x).lower() == '1' or str(x).lower() == 'yes') else 'no')
    else:
        # If no outlier column, just return the original
        return [df.copy()]

    total = len(df_proc)
    current_outliers = len(df_proc[df_proc['outlier'] == 'yes'])
    target = total * float(percent) / 100.0

    if current_outliers <= target:
        # Already within desired percentage: return single copy
        datas.append(df_proc.copy().reset_index(drop=True))
        return datas

    # When there are too many outliers, generate multiple versions by randomly removing outlier rows
    for _ in range(versions):
        df_copy = df_proc.copy(deep=True)
        outlier_indices = df_copy[df_copy['outlier'] == 'yes'].index.tolist()
        num_out = len(outlier_indices)
        # compute how many to remove to reach target (round down)
        remove_count = max(0, int(num_out - target))
        if remove_count > 0 and outlier_indices:
            to_remove = set(random.sample(outlier_indices, k=min(remove_count, len(outlier_indices))))
            df_copy.drop(index=to_remove, inplace=True)
        df_copy.reset_index(drop=True, inplace=True)
        datas.append(df_copy)
        # Update counts so subsequent versions produce different random removals
    return datas


# -------------------------
# I/O helpers
# -------------------------
def save_processed_dataframes(output_dir: str, original_filename: str, dataframes: List[pd.DataFrame]):
    """
    Save each DataFrame in `dataframes` to CSV files inside `output_dir`.
    Filenames are based on original_filename: replace 'raw' with 'processed' if present and append version number.
    """
    os.makedirs(output_dir, exist_ok=True)
    base_name = os.path.basename(original_filename).replace('raw', 'processed')
    count = 1
    for df in dataframes:
        target_name = f"{os.path.splitext(base_name)[0]}_v{count:02d}.csv"
        df.to_csv(os.path.join(output_dir, target_name), index=False)
        count += 1


# -------------------------
# Format converters
# -------------------------
def convert_mat_to_csv(src_dir: str, files: List[str], dst_dir: str = None):
    """
    Convert .mat files containing 'X' (features) and 'y' (labels) to CSV.
    'y' is converted to 'outlier' column with 'yes'/'no' values.
    """
    dst_dir = dst_dir or src_dir
    for f in files:
        mat_path = os.path.join(src_dir, f)
        mat = scipy.io.loadmat(mat_path)
        X = mat.get('X')
        y = mat.get('y')
        if X is None or y is None:
            continue
        cols = [f'attr{i}' for i in range(1, X.shape[1] + 1)]
        data = pd.DataFrame(X, columns=cols)
        # y may be shape (n,1); flatten
        y_flat = y.flatten()
        data['outlier'] = pd.Series(y_flat).apply(lambda v: 'yes' if int(v) == 1 else 'no')
        out_path = os.path.join(dst_dir, f.replace('.mat', '.csv'))
        data.to_csv(out_path, index=False)


def convert_npz_to_csv(src_dir: str, files: List[str], dst_dir: str = None):
    """
    Convert .npz files that contain arrays 'X' and 'y' to CSV files.
    """
    dst_dir = dst_dir or src_dir
    for f in files:
        npz_path = os.path.join(src_dir, f)
        arr = load(npz_path)
        X = arr.get('X')
        y = arr.get('y')
        if X is None or y is None:
            continue
        cols = [f'attr{i}' for i in range(1, X.shape[1] + 1)]
        data = pd.DataFrame(X, columns=cols)
        data['outlier'] = pd.Series(y).apply(lambda v: 'yes' if int(v) == 1 else 'no')
        out_path = os.path.join(dst_dir, f.replace('.npz', '.csv'))
        data.to_csv(out_path, index=False)


# -------------------------
# Main processing for CSV/ARFF datasets
# -------------------------
def process_datasets(input_dir: str, datasets: List[str], output_dir: str, percent: float = 5.0, versions: int = 10):
    """
    Load each dataset in `datasets` from `input_dir`, normalize features, remove duplicates,
    limit outlier fraction to `percent` producing up to `versions` variations, and save to `output_dir`.
    Supports .arff and .csv files.
    """
    os.makedirs(output_dir, exist_ok=True)
    for dataset in datasets:
        print(f"Processing: {dataset}")
        path = os.path.join(input_dir, dataset)
        if dataset.lower().endswith('.arff'):
            data_raw = arff.loadarff(path)
            df = pd.DataFrame(data_raw[0])
            # decode bytes if necessary (for 'outlier' or other nominal fields)
            for col in df.select_dtypes([object]).columns:
                df[col] = df[col].apply(lambda x: x.decode() if isinstance(x, bytes) else x)
            if 'id' in df.columns:
                df.drop(columns=['id'], inplace=True)
        elif dataset.lower().endswith('.csv'):
            df = pd.read_csv(path)
        else:
            print(f"Skipping unsupported format: {dataset}")
            continue

        df = normalize(df)
        df = remove_duplicates(df)
        dfs = limit_outlier_fraction(df, percent=percent, versions=versions)
        save_processed_dataframes(output_dir, dataset, dfs)


# Create directory structure if needed
os.makedirs(r'..\..\datasets\base_experiments\processed\ADBench', exist_ok=True)
os.makedirs(r'..\..\datasets\base_experiments\processed\literature', exist_ok=True)
os.makedirs(r'..\..\datasets\base_experiments\processed\odds', exist_ok=True)
os.makedirs(r'..\..\datasets\base_experiments\processed\real', exist_ok=True)
os.makedirs(r'..\..\datasets\base_experiments\processed\semantic', exist_ok=True)

# -------------------------
# Example usage (kept as cells in original notebook)
# -------------------------
# Convert some .mat files to CSV (original path used in notebook)
# mat_path = r'C:\Users\pipip\Downloads'
# mat_files = ['ecoli.mat', 'wine.mat', 'vertebral.mat', 'mammography.mat', 'optdigits.mat', 'breastw.mat', 'satellite.mat']
# convert_mat_to_csv(mat_path, mat_files)

# Convert some ADBench .npz files
# adbench_path = r'C:\Users\pipip\Google Drive\Doutorado\Experimento\AnaliseOutliers\Resultados\Arquivos\datasets\ADBench'
# adbench_files = ['MNIST-C_scale.npz', 'MNIST-C_motion_blur.npz', 'MNIST-C_rotate.npz', 'MNIST-C_identity.npz', 'MNIST-C_impulse_noise.npz']
# convert_npz_to_csv(adbench_path, adbench_files)

# Process real datasets: normalize, deduplicate, limit outliers and save
# real_path = r'C:\Users\pipip\Google Drive\Doutorado\Experimento\AnaliseOutliers\Resultados\Arquivos\datasets\real'
# real_datasets = ['cfem.csv', 'potabilidade_agua.csv']
# output_dir = real_path  # or choose another directory
# process_datasets(real_path, real_datasets, output_dir, percent=5.0, versions=10)